Colab Link: https://colab.research.google.com/drive/1QyLpDoIQJmTuHAh9ii8-nPP9ZYwjiPtl?usp=sharing

In [ ]:
# Question 1

import pandas as pd
df = pd.read_csv('house_price_regression_dataset.csv')
print("Dataset shape:", df.shape)
print(df.head(10))
print(df.sample(5))

In [ ]:
#Question 2

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

threshold = len(df) * 0.5
df = df.dropna(thresh=threshold, axis=1)

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy='mean')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

X = df.drop(columns=['House_Price'])
y = df['House_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

In [ ]:
#Question 3

import numpy as np

X = X_train['Square_Footage'].values
y = y_train.values


m = 0.0
c = 0.0
learning_rate = 0.0000001
iterations = 1000
n = float(len(X))


for i in range(iterations):
    y_pred = m * X + c

    D_m = (-2/n) * sum(X * (y - y_pred))
    D_c = (-2/n) * sum(y - y_pred)

    m = m - learning_rate * D_m
    c = c - learning_rate * D_c

print(f"Learned slope (m): {m}")
print(f"Learned intercept (c): {c}")


X_test_val = X_test['Square_Footage'].values
predictions = m * X_test_val + c
print("\nTest set predictions (first 5):", predictions[:5])

In [ ]:
#Question 4

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
categorical_features = ['Neighborhood_Quality']

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

In [ ]:
# Question 5

from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# SGDRegressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', SGDRegressor(random_state=42))
])

#train
pipeline.fit(X_train, y_train)

#predict
y_pred = pipeline.predict(X_test)

# RMSE and R2 score
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse}")
print(f"R² Score: {r2}")

# 1st 10 sample
print("\nPredicted vs Actual (First 10 samples):")
comparison = np.vstack((y_pred[:10], y_test[:10])).T
print(comparison)

In [ ]:
# Question 6

from sklearn.linear_model import LinearRegression

# Multiple Linear Regression Pipeline
mlr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# pipeline fit
mlr_pipeline.fit(X_train, y_train)

# predict
y_pred_mlr = mlr_pipeline.predict(X_test)

# RMSE and R2 score evaluation
rmse_mlr = np.sqrt(mean_squared_error(y_test, y_pred_mlr))
r2_mlr = r2_score(y_test, y_pred_mlr)

print(f"Multiple Linear Regression RMSE: {rmse_mlr}")
print(f"Multiple Linear Regression R² Score: {r2_mlr}")

# 1st 10 sample
print("\nPredicted vs Actual (First 10 samples):")
comparison_mlr = np.vstack((y_pred_mlr[:10], y_test[:10])).T
print(comparison_mlr)

In [ ]:
# Question 7

df_sample = df.sample(n=100, random_state=42)

features = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
X = df_sample[features].values
y = df_sample['House_Price'].values

X_b = np.c_[np.ones((100, 1)), X]

theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)

print("Learned coefficients (theta values):")
print(f"Intercept: {theta[0]}")
print(f"Coefficients for {features}: {theta[1:]}")